# Opion Phase-3 Training (T4 GPU)

**Student model: ~1M-param transformer, distilled from teacher elitev6 on 25.4M decisions.**

## How to use
1. **Runtime → Change runtime type → T4 GPU** (do this first!)
2. Drag-drop `packed.tar` (480MB, from the repo's `opion/data/` — I gave you `/tmp/packed.tar`) into the file panel on the left (wait for the upload to finish)
3. Run cells top-to-bottom. Cell 2 keeps the session alive (`free -h` every 5 min, in background).
4. Cell 3 = **smoke run** (300 steps, ~2 min) — check the loss goes down and metrics print.
5. Cell 4 = **full training** (~30-60 min). Checkpoints save to `/content/ckpt/last.pt` every 1000 steps.
6. Cell 5 shows final quality metrics; cell 6 downloads the trained model.

Loss = cross-entropy against the teacher's **soft labels** (its full mixed strategy: bluff frequencies, raise-size splits) — not just its argmax moves. Data is fully GPU-resident, so the T4 stays maxed.


In [1]:
# 1. GPU check
import torch
assert torch.cuda.is_available(), "NO GPU! Runtime -> Change runtime type -> T4 GPU, then re-run all"
print(torch.cuda.get_device_name(0), "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), "GB")

Tesla T4 | VRAM: 14.6 GB


In [2]:
!gdown 1AgcO3IHlhFc4czzr1fU_EG39Bh7809TA


Downloading...
From (original): https://drive.google.com/uc?id=1AgcO3IHlhFc4czzr1fU_EG39Bh7809TA
From (redirected): https://drive.google.com/uc?id=1AgcO3IHlhFc4czzr1fU_EG39Bh7809TA&confirm=t&uuid=9a898564-b0e0-48fe-a41d-86f88fc1b4bd
To: /content/packed.tar
100% 481M/481M [00:03<00:00, 134MB/s]


In [3]:
# 3. Extract the packed dataset (drag-drop packed.tar first!)
import tarfile, glob, os
assert os.path.exists("packed.tar") or glob.glob("packed/*/acts.npz"), "upload packed.tar first (file panel, left)"
if not glob.glob("packed/*/acts.npz"):
    with tarfile.open("packed.tar") as tf:
        tf.extractall("/content")
shards = sorted(glob.glob("packed/*/acts.npz"))
print(len(shards), "shards found")

/tmp/ipykernel_518/634746852.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall("/content")


0 shards found


In [4]:
import tarfile, glob, os

if not glob.glob("packed/*/*/acts.npz"):
    with tarfile.open("packed.tar") as tf:
        tf.extractall("/content")

shards = sorted(glob.glob("packed/*/*/acts.npz"))
print(len(shards), "shards found")   # should print: 40 shards found

40 shards found


In [5]:
# 4. Model: 1M-param transformer (this exact arch ships as the numpy bot later)
import torch
import torch.nn as nn

N_TOK, N_CTX, N_ACT = 41, 17, 6
VOCAB = 85  # 53 card ids (0=pad) + 32 history tokens

def _map_tokens(t):
    t = t.long()
    return torch.where(t < 53, t, 52 + t.clamp(min=1, max=32))

class Block(nn.Module):
    def __init__(self, d, h, f):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.h = h
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, f)
        self.fc2 = nn.Linear(f, d)
        self.act = nn.GELU()
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q = q.view(B, T, self.h, -1).transpose(1, 2)
        k = k.view(B, T, self.h, -1).transpose(1, 2)
        v = v.view(B, T, self.h, -1).transpose(1, 2)
        att = torch.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), -1)
        x = x + self.proj((att @ v).transpose(1, 2).reshape(B, T, D))
        x = x + self.fc2(self.act(self.fc1(self.ln2(x))))
        return x

class OpionPolicy(nn.Module):
    def __init__(self, d=128, layers=4, heads=4, f=512):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(N_TOK, d)
        self.ctx_proj = nn.Sequential(nn.LayerNorm(N_CTX), nn.Linear(N_CTX, d))
        self.blocks = nn.ModuleList(Block(d, heads, f) for _ in range(layers))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, N_ACT)
    def forward(self, tokens, ctx):
        x = self.emb(_map_tokens(tokens)) + self.pos.weight[None, :N_TOK]
        x = x + self.ctx_proj(ctx.float())[:, None, :]
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln_f(x).mean(dim=1))
    def n_params(self):
        return sum(p.numel() for p in self.parameters())

m = OpionPolicy()
print("params:", f"{m.n_params():,}")

params: 812,584


In [6]:
# 5. Data: packed shards -> GPU-resident tensors (train/val split by seed partition)
import numpy as np

def load_shards(root, parts):
    files = sorted(glob.glob(f"{root}/*/*/acts.npz"))
    cols, n = {}, 0
    for f in files:
        z = np.load(f, allow_pickle=False)
        sel = np.isin(z["part"], parts)
        for k in ("tokens", "ctx", "legal", "p_soft", "act_idx", "street"):
            cols.setdefault(k, []).append(z[k][sel])
        n += int(sel.sum())
    return {k: np.concatenate(v) for k, v in cols.items()}, n

def to_gpu(d):
    return {k: torch.from_numpy(v.astype(np.int64 if k in ("tokens","act_idx","street") else np.float32)).cuda()
            for k, v in d.items()}

tr_raw, n_tr = load_shards("packed", (0,))   # train = 92% of match seeds
va_raw, n_va = load_shards("packed", (1,))   # val   = 4%  (test 4% stays untouched)
va_raw = {k: v[:200_000] for k, v in va_raw.items()}
tr, va = to_gpu(tr_raw), to_gpu(va_raw)
print(f"train={n_tr:,}  val={min(n_va,200_000):,}  (on GPU)")
free, total = torch.cuda.mem_get_info()
print(f"VRAM free: {free>>20}MB / {total>>20}MB")

train=20,343,138  val=200,000  (on GPU)
VRAM free: 5781MB / 14912MB


In [7]:
# 6. Trainer: soft-label distillation + metrics  (fp16-safe masking)
import torch.nn.functional as F
import time, json, os

def evaluate(model, va, batch=16384):
    model.eval()
    kl_s = tv_s = ag_s = ah_s = n = 0.0
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(va["act_idx"]), batch):
            sl = slice(i, i + batch)
            logits = model(va["tokens"][sl], va["ctx"][sl]).float()      # FIX: float() first
            logits = logits.masked_fill(va["legal"][sl] == 0, -1e9)      # now safe
            q = torch.softmax(logits.float(), -1)
            p = va["p_soft"][sl].float()
            kl = (p * (torch.log(p + 1e-8) - torch.log(q + 1e-8))).sum(-1).mean().item()
            tv = (0.5 * (p - q).abs().sum(-1)).mean().item()
            ag = (p.argmax(-1) == q.argmax(-1)).float().mean().item()
            b = len(p)
            kl_s += kl*b; tv_s += tv*b; ag_s += ag*b
            ah_s += (q.argmax(-1) == va["act_idx"][sl]).float().sum().item(); n += b
    model.train()
    return kl_s/n, tv_s/n, ag_s/n, ah_s/n

def run_training(epochs=3.0, batch=4096, lr=3e-4, max_steps=0, out="/content/ckpt"):
    os.makedirs(out, exist_ok=True)
    model = OpionPolicy().cuda()
    print("params:", f"{model.n_params():,}")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = int((n_tr // batch) * epochs)
    if max_steps: total = min(total, max_steps)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=total, pct_start=0.05)
    scaler = torch.amp.GradScaler("cuda")                            # modern API (kills the warning too)
    print(f"steps={total}  batch={batch}")
    t0 = time.time()
    for step in range(1, total + 1):
        idx = torch.randint(0, n_tr, (batch,), device="cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(tr["tokens"][idx], tr["ctx"][idx]).float()  # FIX: float() first
            logits = logits.masked_fill(tr["legal"][idx] == 0, -1e9)   # now safe
            logq = torch.log_softmax(logits, -1)
            loss = -(tr["p_soft"][idx].float() * logq).sum(-1).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        if step % 100 == 0 or step == total:
            with torch.no_grad():
                q = torch.softmax(logits, -1)
                ag = (q.argmax(-1) == tr["act_idx"][idx]).float().mean()
            print(f"step {step}/{total}  loss={loss.item():.4f}  hard_ag={ag.item():.3f}  "
                  f"lr={sched.get_last_lr()[0]:.1e}  {time.time()-t0:.0f}s", flush=True)
        if step % 1000 == 0 or step == total:
            kl, tv, ag, ah = evaluate(model, va)
            print(f"  VAL step {step}: KL={kl:.4f} TV={tv:.4f} argmax_ag={ag:.3f} hard_ag={ah:.3f}", flush=True)
            torch.save({"model": model.state_dict(), "step": step}, f"{out}/last.pt")
    kl, tv, ag, ah = evaluate(model, va)
    final = dict(kl=kl, tv=tv, argmax_ag=ag, hard_ag=ah, steps=total, wall_min=round((time.time()-t0)/60,1))
    torch.save({"model": model.state_dict(), "final": final}, f"{out}/final.pt")
    print("FINAL:", json.dumps(final))
    return model, final

print("trainer ready")

trainer ready


In [8]:
# 7. SMOKE RUN (300 steps, ~2 min) — verify loss drops + metrics print
model_smoke, final_smoke = run_training(epochs=1.0, batch=512, max_steps=300, out="/content/ckpt_smoke")
print("SMOKE_OK" if final_smoke["hard_ag"] > 0.3 else "SMOKE_SUSPICIOUS: check loss curve above")

params: 812,584


KeyboardInterrupt: 

In [9]:
# 8. FULL TRAINING (~30-60 min on T4) — run after smoke passes
model, final = run_training(epochs=3.0, batch=4096, lr=3e-4, out="/content/ckpt")

params: 812,584
steps=14898  batch=4096
step 100/14898  loss=0.9372  hard_ag=0.660  lr=2.5e-05  23s
step 200/14898  loss=0.4887  hard_ag=0.792  lr=6.0e-05  46s
step 300/14898  loss=0.3664  hard_ag=0.826  lr=1.1e-04  70s
step 400/14898  loss=0.2959  hard_ag=0.873  lr=1.7e-04  93s
step 500/14898  loss=0.2621  hard_ag=0.901  lr=2.3e-04  117s
step 600/14898  loss=0.2368  hard_ag=0.907  lr=2.7e-04  141s
step 700/14898  loss=0.2385  hard_ag=0.910  lr=3.0e-04  165s
step 800/14898  loss=0.2179  hard_ag=0.917  lr=3.0e-04  189s
step 900/14898  loss=0.2422  hard_ag=0.898  lr=3.0e-04  213s
step 1000/14898  loss=0.2027  hard_ag=0.929  lr=3.0e-04  238s
  VAL step 1000: KL=0.2047 TV=0.1294 argmax_ag=0.913 hard_ag=0.918
step 1100/14898  loss=0.2091  hard_ag=0.929  lr=3.0e-04  267s
step 1200/14898  loss=0.2083  hard_ag=0.925  lr=3.0e-04  291s
step 1300/14898  loss=0.1880  hard_ag=0.934  lr=3.0e-04  316s
step 1400/14898  loss=0.1860  hard_ag=0.931  lr=3.0e-04  340s
step 1500/14898  loss=0.1856  hard_ag=

In [10]:
# 9. Final quality report (what the numbers mean):
# KL   = how far the student's probabilities are from the teacher's (lower=better; ~<0.05 is excellent)
# TV   = total variation, half the L1 gap between distributions (lower=better)
# argmax_ag = student's best move == teacher's best move
# hard_ag   = student's sampled-with-teacher-temperature agreement (rough play parity)
kl, tv, ag, ah = evaluate(model, va)
print(f"KL={kl:.4f}  TV={tv:.4f}  argmax_ag={ag:.3f}  hard_ag={ah:.3f}")

KL=0.0901  TV=0.0551  argmax_ag=0.965  hard_ag=0.971


In [11]:
# 10. Download the trained model (or grab it later from /content/ckpt/)
from google.colab import files
files.download("/content/ckpt/final.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>